# 03 — Task 2.3: Fraud Feature Engineering

**Target Table:** `bfsi.silver_layer.transaction_features`

**Objective:** Compute time-windowed fraud detection features on top of `unified_transactions`.

| Feature | Window | Description |
|---------|--------|-------------|
| `txn_count_1h` | 1 hour | Customer's transaction count in past 1 hour |
| `amount_sum_1h` | 1 hour | Total amount spent in past 1 hour |
| `distinct_merchants_1h` | 1 hour | Unique merchant_ids in past 1 hour |
| `distinct_countries_24h` | 24 hours | Unique countries in past 24 hours (impossible travel) |
| `time_since_last_txn_seconds` | — | Seconds since customer's previous transaction |
| `amount_zscore` | 30 days | `(amount - 30d_avg) / 30d_std` — deviation from norm |

> Reads from `silver.unified_transactions` (Task 2.2 output).

---
## Cell 1 — Configuration & Imports

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.types import LongType, DoubleType
from datetime import datetime

# ── Catalog / Schema ───────────────────────────────────────────────────
CATALOG       = "bfsi"
SILVER_SCHEMA = "silver_layer"

# ── Source table (Task 2.2 output) ─────────────────────────────────────
UNIFIED_TABLE  = f"{CATALOG}.{SILVER_SCHEMA}.unified_transactions"

# ── Target table ───────────────────────────────────────────────────────
FEATURES_TABLE = f"{CATALOG}.{SILVER_SCHEMA}.transaction_features"

# ── Window durations (seconds) ─────────────────────────────────────────
ONE_HOUR_SEC   = 3600
TWENTY_FOUR_H  = 86400
THIRTY_DAYS    = 30 * 86400   # 2,592,000 seconds

# ── Incremental: get watermark from target table ───────────────────────
try:
    _max_ts = (
        spark.read.table(FEATURES_TABLE)
        .agg(F.max("_silver_load_ts").alias("max_ts"))
        .collect()[0]["max_ts"]
    )
    LAST_LOAD_TS = _max_ts if _max_ts is not None else datetime(1900, 1, 1)
except Exception:
    LAST_LOAD_TS = datetime(1900, 1, 1)

print(f"Task 2.3: Fraud Feature Engineering (INCREMENTAL)")
print(f"Source    : {UNIFIED_TABLE}")
print(f"Target    : {FEATURES_TABLE}")
print(f"Watermark : {LAST_LOAD_TS}")
print(f"Timestamp : {datetime.now().isoformat()}")

---
## Cell 2 — Load Unified Transactions

In [0]:
# ══════════════════════════════════════════════════════════════════════════
#  LOAD unified_transactions FROM SILVER (INCREMENTAL)
#  Only loads new records since the last processing run.
# ══════════════════════════════════════════════════════════════════════════

df_unified = (
    spark.read.table(UNIFIED_TABLE)
    .filter(F.col("_silver_load_ts") > F.lit(LAST_LOAD_TS))
)

new_count = df_unified.count()
print(f"New records to process: {new_count:,}")

if new_count == 0:
    print("No new data found. Skipping feature engineering.")
    dbutils.notebook.exit("NO_NEW_DATA")

print(f"Columns: {df_unified.columns}")
print(f"\nSchema:")
df_unified.printSchema()

---
## Cell 3 — Define Window Specifications

In [0]:
# ══════════════════════════════════════════════════════════════════════════
#  WINDOW SPECIFICATIONS
#  All windows are partitioned by customer_id, ordered by txn_ts
#  Range-based windows use seconds for time-based rolling calculations
# ══════════════════════════════════════════════════════════════════════════

# Convert txn_ts to unix timestamp for range-based windows
df_with_epoch = df_unified.withColumn(
    "_txn_epoch", F.unix_timestamp("txn_ts")
)

# ── 1-hour rolling window (per customer) ───────────────────────────────
# Includes current row and all rows within 1 hour BEFORE current txn
window_1h = (
    Window
    .partitionBy("customer_id")
    .orderBy("_txn_epoch")
    .rangeBetween(-ONE_HOUR_SEC, 0)     # past 1 hour to current row
)

# ── 24-hour rolling window (per customer) ──────────────────────────────
window_24h = (
    Window
    .partitionBy("customer_id")
    .orderBy("_txn_epoch")
    .rangeBetween(-TWENTY_FOUR_H, 0)    # past 24 hours to current row
)

# ── 30-day rolling window (per customer) ───────────────────────────────
window_30d = (
    Window
    .partitionBy("customer_id")
    .orderBy("_txn_epoch")
    .rangeBetween(-THIRTY_DAYS, 0)      # past 30 days to current row
)

# ── Unbounded previous row window (for time_since_last) ────────────────
window_prev = (
    Window
    .partitionBy("customer_id")
    .orderBy("_txn_epoch")
)

print("Window specifications defined:")
print("  window_1h   — 1-hour rolling (range-based)")
print("  window_24h  — 24-hour rolling (range-based)")
print("  window_30d  — 30-day rolling (range-based)")
print("  window_prev — row-based (lag for previous txn)")

---
## Cell 4 — Feature: txn_count_1h

In [0]:
# ══════════════════════════════════════════════════════════════════════════
#  FEATURE 1: txn_count_1h
#  Number of transactions for this customer in the past 1 hour
#  High values → rapid-fire fraud pattern (card testing, bot attacks)
# ══════════════════════════════════════════════════════════════════════════

df_feat = df_with_epoch.withColumn(
    "txn_count_1h",
    F.count("txn_id").over(window_1h)
)

print("Feature 1: txn_count_1h — computed")
print(f"  Max txn_count_1h: {df_feat.agg(F.max('txn_count_1h')).collect()[0][0]}")
print(f"  Avg txn_count_1h: {df_feat.agg(F.round(F.avg('txn_count_1h'), 2)).collect()[0][0]}")

---
## Cell 5 — Feature: amount_sum_1h

In [0]:
# ══════════════════════════════════════════════════════════════════════════
#  FEATURE 2: amount_sum_1h
#  Total amount spent by this customer in the past 1 hour (INR)
#  High values → large burst spending, potential account takeover
# ══════════════════════════════════════════════════════════════════════════

df_feat = df_feat.withColumn(
    "amount_sum_1h",
    F.round(F.sum("txn_amount_inr").over(window_1h), 2)
)

print("Feature 2: amount_sum_1h — computed")
print(f"  Max amount_sum_1h: ₹{df_feat.agg(F.max('amount_sum_1h')).collect()[0][0]:,.2f}")
print(f"  Avg amount_sum_1h: ₹{df_feat.agg(F.round(F.avg('amount_sum_1h'), 2)).collect()[0][0]:,.2f}")

---
## Cell 6 — Feature: distinct_merchants_1h

In [0]:
# ══════════════════════════════════════════════════════════════════════════
#  FEATURE 3: distinct_merchants_1h
#  Unique merchant_ids this customer transacted with in past 1 hour
#  High values → card testing across multiple merchants
#
#  Note: Spark window functions do not support COUNT(DISTINCT) directly.
#  We use collect_set + size as the equivalent.
# ══════════════════════════════════════════════════════════════════════════

df_feat = df_feat.withColumn(
    "distinct_merchants_1h",
    F.size(
        F.collect_set("merchant_id").over(window_1h)
    )
)

# If merchant_id is NULL (UPI/NEFT), collect_set will have null entries;
# size counts them as 0 or 1 (null element). Fix: filter nulls.
# collect_set automatically excludes nulls in Spark, so this is clean.

print("Feature 3: distinct_merchants_1h — computed")
print(f"  Max distinct_merchants_1h: {df_feat.agg(F.max('distinct_merchants_1h')).collect()[0][0]}")

---
## Cell 10 — Write Transaction Features to Silver

In [0]:
# ══════════════════════════════════════════════════════════════════════════
#  FEATURE 4: distinct_countries_24h
#  Unique countries this customer transacted in during the past 24 hours
#  Values > 2 → impossible travel detection (fraud signal)
#
#  For Card txns: uses original_currency as a proxy for country
#  (INR=India, USD=US, GBP=UK, etc.)
#  For UPI/NEFT: always "INR" (domestic only)
# ══════════════════════════════════════════════════════════════════════════

df_feat = df_feat.withColumn(
    "distinct_countries_24h",
    F.size(
        F.collect_set("original_currency").over(window_24h)
    )
)

print("Feature 4: distinct_countries_24h — computed")
print(f"  Max distinct_countries_24h: {df_feat.agg(F.max('distinct_countries_24h')).collect()[0][0]}")

# Flag impossible travel (>2 countries in 24h)
impossible_travel = df_feat.filter(F.col("distinct_countries_24h") > 2).count()
print(f"  Impossible travel candidates (>2 countries in 24h): {impossible_travel:,}")

---
## Cell 8 — Feature: time_since_last_txn_seconds

In [0]:
# ══════════════════════════════════════════════════════════════════════════
#  FEATURE 5: time_since_last_txn_seconds
#  Seconds elapsed since this customer's previous transaction
#  Very small values → automated fraud (bot, card testing)
#  NULL for customer's first transaction
# ══════════════════════════════════════════════════════════════════════════

df_feat = df_feat.withColumn(
    "prev_txn_epoch",
    F.lag("_txn_epoch", 1).over(window_prev)
).withColumn(
    "time_since_last_txn_seconds",
    F.when(
        F.col("prev_txn_epoch").isNotNull(),
        (F.col("_txn_epoch") - F.col("prev_txn_epoch")).cast(LongType())
    ).otherwise(F.lit(None).cast(LongType()))
).drop("prev_txn_epoch")

print("Feature 5: time_since_last_txn_seconds — computed")
print(f"  Min gap (excl. nulls): {df_feat.filter(F.col('time_since_last_txn_seconds').isNotNull()).agg(F.min('time_since_last_txn_seconds')).collect()[0][0]}s")
print(f"  Avg gap: {df_feat.agg(F.round(F.avg('time_since_last_txn_seconds'), 0)).collect()[0][0]}s")

# Flag rapid-fire (< 60 seconds between txns)
rapid_fire = df_feat.filter(F.col("time_since_last_txn_seconds") < 60).count()
print(f"  Rapid-fire transactions (<60s gap): {rapid_fire:,}")

---
## Cell 9 — Feature: amount_zscore

In [0]:
# ══════════════════════════════════════════════════════════════════════════
#  FEATURE 6: amount_zscore
#  Z-score of transaction amount relative to customer's 30-day behavior:
#    z = (txn_amount_inr - avg_30d) / stddev_30d
#
#  |z| > 3  → highly anomalous (fraud candidate)
#  |z| > 2  → unusual (worth monitoring)
#  NULL if customer has < 2 transactions in 30d (can't compute stddev)
# ══════════════════════════════════════════════════════════════════════════

df_feat = (
    df_feat
    # 30-day rolling average
    .withColumn(
        "_avg_30d",
        F.avg("txn_amount_inr").over(window_30d)
    )
    # 30-day rolling standard deviation
    .withColumn(
        "_std_30d",
        F.stddev("txn_amount_inr").over(window_30d)
    )
    # Z-score calculation
    .withColumn(
        "amount_zscore",
        F.when(
            # Avoid division by zero / null stddev (< 2 data points)
            (F.col("_std_30d").isNotNull()) & (F.col("_std_30d") > 0),
            F.round(
                (F.col("txn_amount_inr") - F.col("_avg_30d")) / F.col("_std_30d"),
                4
            )
        ).otherwise(F.lit(None).cast(DoubleType()))
    )
    # Drop intermediate columns
    .drop("_avg_30d", "_std_30d")
)

print("Feature 6: amount_zscore — computed")
print(f"  Max |z-score|: {df_feat.agg(F.max(F.abs('amount_zscore'))).collect()[0][0]}")

# Flag anomalous transactions
z_above_3 = df_feat.filter(F.abs(F.col("amount_zscore")) > 3).count()
z_above_2 = df_feat.filter(F.abs(F.col("amount_zscore")) > 2).count()
print(f"  |z| > 3 (highly anomalous): {z_above_3:,}")
print(f"  |z| > 2 (unusual)         : {z_above_2:,}")

---
## Cell 10 — Write Transaction Features to Silver

In [0]:
# ══════════════════════════════════════════════════════════════════════════
#  WRITE transaction_features TO SILVER DELTA TABLE (INCREMENTAL APPEND)
#  Includes all original unified_transactions columns + 6 fraud features
# ══════════════════════════════════════════════════════════════════════════

# Final cleanup: drop internal epoch column
df_final = (
    df_feat
    .drop("_txn_epoch")
    .withColumn("_silver_load_ts", F.current_timestamp())
    .withColumn("_feature_version", F.lit("v1.0-task2.3"))
)

# Append new records to Delta
df_final.write \
    .format("delta") \
    .mode("append") \
    .saveAsTable(FEATURES_TABLE)

print(f"\n Silver table updated: {FEATURES_TABLE}")
print(f"New records appended: {new_count:,}")
print(f"Total records: {spark.read.table(FEATURES_TABLE).count():,}")

---
## Cell 11 — Verification & Feature Summary

In [0]:
# ══════════════════════════════════════════════════════════════════════════
#  TASK 2.3 — VERIFICATION & FEATURE STATISTICS
# ══════════════════════════════════════════════════════════════════════════

df_result = spark.read.table(FEATURES_TABLE)

print("=" * 70)
print("  TASK 2.3: FRAUD FEATURE ENGINEERING — COMPLETE")
print("=" * 70)

# ── Feature statistics ──
print("\n Feature Statistics:")
feature_cols = [
    "txn_count_1h",
    "amount_sum_1h",
    "distinct_merchants_1h",
    "distinct_countries_24h",
    "time_since_last_txn_seconds",
    "amount_zscore",
]

print(f"\n{'Feature':<35} {'Min':>12} {'Avg':>12} {'Max':>12} {'Nulls':>10}")
print("-" * 83)

for feat in feature_cols:
    stats = df_result.agg(
        F.min(feat).alias("min_val"),
        F.round(F.avg(feat), 2).alias("avg_val"),
        F.max(feat).alias("max_val"),
        F.sum(F.when(F.col(feat).isNull(), 1).otherwise(0)).alias("null_count"),
    ).collect()[0]
    print(
        f"{feat:<35} {str(stats['min_val']):>12} "
        f"{str(stats['avg_val']):>12} {str(stats['max_val']):>12} "
        f"{str(stats['null_count']):>10}"
    )

# ── Schema validation ──
expected_features = feature_cols + [
    "txn_id", "customer_id", "txn_channel", "txn_amount_inr",
    "txn_ts", "status", "masked_counterparty", "merchant_id",
    "is_international"
]
actual_cols = df_result.columns
missing = [c for c in expected_features if c not in actual_cols]

print(f"\n Schema validation: {'ALL columns present' if not missing else f'MISSING: {missing}'}")
print(f" Total columns: {len(actual_cols)}")
print(f" Total records: {df_result.count():,}")

# ── Fraud risk indicators ──
print("\n Fraud Risk Indicators:")
print(f"  ▸ Rapid-fire txns (<60s gap)    : {df_result.filter(F.col('time_since_last_txn_seconds') < 60).count():,}")
print(f"  ▸ High velocity (>5 txns/hr)    : {df_result.filter(F.col('txn_count_1h') > 5).count():,}")
print(f"  ▸ Impossible travel (>2 ctry/24h): {df_result.filter(F.col('distinct_countries_24h') > 2).count():,}")
print(f"  ▸ Amount anomaly (|z| > 3)      : {df_result.filter(F.abs(F.col('amount_zscore')) > 3).count():,}")

print(f"\n Timestamp: {datetime.now().isoformat()}")
print("=" * 70)

In [0]:
# ── Sample output: Top suspicious transactions ─────────────────────────
print("Top 10 transactions by z-score (most anomalous):")
display(
    df_result
    .filter(F.col("amount_zscore").isNotNull())
    .select(
        "txn_id", "customer_id", "txn_channel",
        "txn_amount_inr", "txn_ts", "status",
        "txn_count_1h", "amount_sum_1h",
        "distinct_merchants_1h", "distinct_countries_24h",
        "time_since_last_txn_seconds", "amount_zscore",
    )
    .orderBy(F.abs(F.col("amount_zscore")).desc())
    .limit(10)
)

---

### Feature Descriptions

| Feature | Type | Fraud Signal | Threshold |
|---------|------|-------------|------------|
| `txn_count_1h` | INT | Rapid-fire card testing | > 5 txns/hr |
| `amount_sum_1h` | DOUBLE | Burst spending / account takeover | Context-dependent |
| `distinct_merchants_1h` | INT | Testing card across merchants | > 3 merchants/hr |
| `distinct_countries_24h` | INT | Impossible travel | > 2 countries/24h |
| `time_since_last_txn_seconds` | LONG | Bot / automated fraud | < 60 seconds |
| `amount_zscore` | DOUBLE | Unusual spending deviation | \|z\| > 3 |

> **Next:** Gold Layer — Fraud Alerts, SAR reports, Risk Scoring